# 04 — Reject Inference: Correcting Selection Bias

When a lender only observes outcomes for approved applicants, the training data suffers from **selection bias**.  
Rejected applicants — who may have defaulted at higher rates — are invisible to the model.

**Approach: Parcelling (Simple Augmentation)**  
1. Train a model on accepted loans  
2. Score rejected applicants with it  
3. Assign pseudo-labels based on score (high score → likely default)  
4. Retrain on the combined dataset with soft labels

This is a simplified approach. Production methods include EM-based techniques and Heckman correction.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve

plt.style.use('seaborn-v0_8-whitegrid')

SILVER_DIR = Path("../data/silver")
GOLD_DIR = Path("../data/gold")
MODELS_DIR = Path("../data/models")

# Load accepted (Gold) data
with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)
feature_cols = meta["feature_columns"]

train = pd.read_parquet(GOLD_DIR / "features_train.parquet")
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")

# Load rejected (Silver) data
rejected = pd.read_parquet(SILVER_DIR / "rejected_clean.parquet")

# Load champion model (trained on accepted only)
with open(MODELS_DIR / "champion" / "model.pkl", "rb") as f:
    accepted_model = pickle.load(f)

print(f"Accepted train: {len(train):,} rows")
print(f"Rejected: {len(rejected):,} rows")
print(f"\nAccepted features: {feature_cols[:5]}...")
print(f"Rejected features: {rejected.columns.tolist()}")

## 1. Align Rejected Data to Accepted Feature Space

Rejected applicants have fewer columns. We use the overlap (FICO, DTI, loan amount, emp_length) and fill the rest with population medians from the accepted training set.

In [ ]:
# Sample rejected to avoid memory issues (take 500K)
np.random.seed(42)
rejected_sample = rejected.sample(n=min(500_000, len(rejected)), random_state=42).copy()

# Build a feature-aligned dataframe for rejected applicants
# Overlapping features: fico_score, dti, loan_amnt, emp_length, emp_length_missing
rejected_aligned = pd.DataFrame(index=rejected_sample.index)

# Direct mappings
rejected_aligned['fico_score'] = rejected_sample['fico_score']
rejected_aligned['dti'] = rejected_sample['dti']
rejected_aligned['loan_amnt'] = rejected_sample['loan_amnt']
rejected_aligned['emp_length'] = rejected_sample['emp_length']
rejected_aligned['emp_length_missing'] = rejected_sample['emp_length_missing']

# Fill remaining features with training set medians (conservative imputation)
train_medians = train[feature_cols].median()
for col in feature_cols:
    if col not in rejected_aligned.columns:
        rejected_aligned[col] = train_medians[col]

# Ensure column order matches
rejected_aligned = rejected_aligned[feature_cols]

print(f"Rejected aligned: {rejected_aligned.shape}")
print(f"Null check: {rejected_aligned.isnull().sum().sum()} nulls")

# Fill any remaining NaN
rejected_aligned = rejected_aligned.fillna(train_medians)

## 2. Score Rejected Applicants & Assign Pseudo-Labels

In [ ]:
# Score rejected applicants using the accepted-only model
rejected_scores = accepted_model.predict_proba(rejected_aligned)[:, 1]

# Parcelling: assign hard pseudo-labels based on score
# Use a higher threshold than the training default rate (rejected pop is riskier)
training_default_rate = train['default'].mean()
print(f"Training default rate (accepted): {training_default_rate:.3f}")

# Rejected applicants were turned down — assume they would default at ~2x the accepted rate
reject_default_rate = min(training_default_rate * 2.0, 0.60)
threshold = np.percentile(rejected_scores, 100 * (1 - reject_default_rate))
print(f"Assumed reject default rate: {reject_default_rate:.3f}")
print(f"Score threshold for pseudo-default: {threshold:.4f}")

rejected_aligned['default'] = (rejected_scores >= threshold).astype(int)

print(f"\nPseudo-label distribution:")
print(f"  Non-default: {(rejected_aligned['default'] == 0).sum():,}")
print(f"  Default:     {(rejected_aligned['default'] == 1).sum():,}")
print(f"  Rate:        {rejected_aligned['default'].mean():.3f}")

# Compare score distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(accepted_model.predict_proba(train[feature_cols])[:, 1], bins=50, alpha=0.5, 
        density=True, label='Accepted (known outcomes)', color='#2ecc71')
ax.hist(rejected_scores, bins=50, alpha=0.5, 
        density=True, label='Rejected (pseudo-labels)', color='#e74c3c')
ax.axvline(threshold, color='black', linestyle='--', label=f'Pseudo-label threshold ({threshold:.3f})')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution: Accepted vs Rejected Applicants')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Retrain on Combined Dataset & Compare

In [ ]:
# Combine accepted + rejected (with pseudo-labels)
# Weight rejected samples lower (0.3) since labels are uncertain
X_combined = pd.concat([train[feature_cols], rejected_aligned[feature_cols]], ignore_index=True)
y_combined = pd.concat([train['default'], rejected_aligned['default']], ignore_index=True)

sample_weights = np.concatenate([
    np.ones(len(train)),                      # accepted: weight 1.0
    np.full(len(rejected_aligned), 0.3),      # rejected: weight 0.3 (uncertain labels)
])

print(f"Combined dataset: {len(X_combined):,} rows")
print(f"  Accepted: {len(train):,} (weight=1.0)")
print(f"  Rejected: {len(rejected_aligned):,} (weight=0.3)")

# Train augmented model
augmented_model = HistGradientBoostingClassifier(
    max_iter=1000,
    max_depth=6,
    learning_rate=0.05,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=50,
    scoring="roc_auc",
    random_state=42,
    verbose=0,
)

augmented_model.fit(X_combined, y_combined, sample_weight=sample_weights)
print(f"Augmented model iterations: {augmented_model.n_iter_}")

## 4. Compare: Accepted-Only vs Augmented Model

In [ ]:
def compute_ks(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

X_test = test[feature_cols]
y_test = test['default']

# Score test set with both models
scores_accepted = accepted_model.predict_proba(X_test)[:, 1]
scores_augmented = augmented_model.predict_proba(X_test)[:, 1]

auc_acc = roc_auc_score(y_test, scores_accepted)
auc_aug = roc_auc_score(y_test, scores_augmented)
ks_acc = compute_ks(y_test, scores_accepted)
ks_aug = compute_ks(y_test, scores_augmented)

print("Model Comparison on Hold-Out Test Set")
print("=" * 50)
print(f"{'Metric':<20} {'Accepted-Only':>15} {'Augmented':>15}")
print("-" * 50)
print(f"{'AUC':.<20} {auc_acc:>15.4f} {auc_aug:>15.4f}")
print(f"{'KS':.<20} {ks_acc:>15.4f} {ks_aug:>15.4f}")
print(f"{'Gini':.<20} {2*auc_acc-1:>15.4f} {2*auc_aug-1:>15.4f}")

# ROC curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fpr_a, tpr_a, _ = roc_curve(y_test, scores_accepted)
fpr_g, tpr_g, _ = roc_curve(y_test, scores_augmented)

axes[0].plot(fpr_a, tpr_a, label=f'Accepted-Only (AUC={auc_acc:.4f})', color='#3498db', linewidth=2)
axes[0].plot(fpr_g, tpr_g, label=f'Augmented (AUC={auc_aug:.4f})', color='#e74c3c', linewidth=2)
axes[0].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve Comparison'); axes[0].legend()

# Score distributions
axes[1].hist(scores_accepted, bins=50, alpha=0.5, density=True, label='Accepted-Only', color='#3498db')
axes[1].hist(scores_augmented, bins=50, alpha=0.5, density=True, label='Augmented', color='#e74c3c')
axes[1].set_xlabel('Predicted PD'); axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution on Test Set'); axes[1].legend()

# Calibration comparison
from sklearn.calibration import calibration_curve
for scores, name, color in [(scores_accepted, 'Accepted-Only', '#3498db'), 
                              (scores_augmented, 'Augmented', '#e74c3c')]:
    prob_true, prob_pred = calibration_curve(y_test, scores, n_bins=10)
    axes[2].plot(prob_pred, prob_true, 's-', color=color, label=name, linewidth=2)
axes[2].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[2].set_xlabel('Predicted Probability'); axes[2].set_ylabel('Actual Probability')
axes[2].set_title('Calibration Comparison'); axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Decile Analysis — Rank Ordering Check

In [ ]:
# Decile analysis for both models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, scores, name in [(axes[0], scores_accepted, 'Accepted-Only'),
                           (axes[1], scores_augmented, 'Augmented')]:
    decile_df = pd.DataFrame({'score': scores, 'default': y_test.values})
    decile_df['decile'] = pd.qcut(decile_df['score'], 10, labels=range(1, 11))
    
    stats = decile_df.groupby('decile', observed=True).agg(
        default_rate=('default', 'mean'),
        count=('default', 'size'),
    ).reset_index()
    
    bars = ax.bar(stats['decile'].astype(str), stats['default_rate'],
                  color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)), edgecolor='black')
    for bar, rate in zip(bars, stats['default_rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{rate:.1%}', ha='center', fontsize=8)
    ax.set_xlabel('Score Decile (1=lowest risk)')
    ax.set_ylabel('Default Rate')
    ax.set_title(f'{name} Model — Decile Default Rates')
    
    # Check rank ordering
    rates = stats['default_rate'].values
    breaks = sum(1 for i in range(1, len(rates)) if rates[i] < rates[i-1])
    ax.text(0.02, 0.95, f'Rank-order breaks: {breaks}', transform=ax.transAxes,
            fontweight='bold', fontsize=10, va='top')

plt.tight_layout()
plt.show()

## 6. Feature Importance Shift — What Changed?

In [ ]:
# Compare permutation importances (more reliable than tree-based importance for comparison)
from sklearn.inspection import permutation_importance

# Use a subset for speed
np.random.seed(42)
idx = np.random.choice(len(X_test), size=min(5000, len(X_test)), replace=False)
X_sub = X_test.iloc[idx]
y_sub = y_test.iloc[idx]

perm_acc = permutation_importance(accepted_model, X_sub, y_sub, n_repeats=5, 
                                   scoring='roc_auc', random_state=42, n_jobs=-1)
perm_aug = permutation_importance(augmented_model, X_sub, y_sub, n_repeats=5,
                                   scoring='roc_auc', random_state=42, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'accepted_only': perm_acc.importances_mean,
    'augmented': perm_aug.importances_mean,
})
imp_df['shift'] = imp_df['augmented'] - imp_df['accepted_only']
imp_df = imp_df.sort_values('accepted_only', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(imp_df))
width = 0.35
ax.barh(x + width/2, imp_df['accepted_only'], width, label='Accepted-Only', color='#3498db', edgecolor='black')
ax.barh(x - width/2, imp_df['augmented'], width, label='Augmented', color='#e74c3c', edgecolor='black')
ax.set_yticks(x)
ax.set_yticklabels(imp_df['feature'])
ax.set_xlabel('Permutation Importance (AUC drop)')
ax.set_title('Feature Importance Shift After Reject Inference')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Show biggest shifts
print("Biggest importance shifts (augmented - accepted):")
shift_df = pd.DataFrame({
    'feature': feature_cols,
    'accepted_only': perm_acc.importances_mean,
    'augmented': perm_aug.importances_mean,
})
shift_df['shift'] = shift_df['augmented'] - shift_df['accepted_only']
shift_df['pct_change'] = (shift_df['shift'] / shift_df['accepted_only'].clip(lower=1e-6) * 100)
print(shift_df.sort_values('shift', key=abs, ascending=False).head(10).to_string(index=False))

## 7. PSI Between Models — Score Stability

Check if the augmented model produces a fundamentally different score distribution (Population Stability Index).

In [ ]:
def compute_psi(expected, actual, bins=10):
    """Population Stability Index between two score distributions."""
    bin_edges = np.linspace(0, 1, bins + 1)
    exp_counts, _ = np.histogram(expected, bins=bin_edges)
    act_counts, _ = np.histogram(actual, bins=bin_edges)
    
    exp_pct = exp_counts / exp_counts.sum()
    act_pct = act_counts / act_counts.sum()
    
    # Avoid log(0)
    exp_pct = np.clip(exp_pct, 1e-6, None)
    act_pct = np.clip(act_pct, 1e-6, None)
    
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi, bin_edges, exp_pct, act_pct

psi, edges, pct_acc, pct_aug = compute_psi(scores_accepted, scores_augmented, bins=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSI bar comparison
midpoints = (edges[:-1] + edges[1:]) / 2
width = (edges[1] - edges[0]) * 0.4
axes[0].bar(midpoints - width/2, pct_acc, width, label='Accepted-Only', color='#3498db', edgecolor='black')
axes[0].bar(midpoints + width/2, pct_aug, width, label='Augmented', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Score Bin')
axes[0].set_ylabel('Proportion')
axes[0].set_title(f'Score Distribution Comparison (PSI = {psi:.4f})')
axes[0].legend()

# Per-bin PSI contribution
per_bin_psi = (pct_aug - pct_acc) * np.log(pct_aug / pct_acc)
colors = ['#e74c3c' if p > 0.02 else '#f39c12' if p > 0.01 else '#2ecc71' for p in per_bin_psi]
axes[1].bar(range(1, 11), per_bin_psi, color=colors, edgecolor='black')
axes[1].set_xlabel('Score Bin')
axes[1].set_ylabel('PSI Contribution')
axes[1].set_title('Per-Bin PSI Contribution')
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"Overall PSI: {psi:.4f}")
if psi < 0.10:
    print("Interpretation: Minimal shift — models produce similar score distributions.")
elif psi < 0.25:
    print("Interpretation: Moderate shift — investigate which score bands changed.")
else:
    print("Interpretation: Significant shift — augmented model behaves very differently.")

## 8. Conclusion

**Key Takeaways:**

1. **Selection bias is real** — the accepted-only model has never seen the risk profiles of rejected applicants, creating a blind spot in the score distribution.

2. **Parcelling is a simple but effective correction** — by scoring rejected applicants with the existing model and assigning pseudo-labels at a higher assumed default rate, we expose the model to a broader applicant population.

3. **Sample weighting is critical** — rejected pseudo-labels are uncertain, so we weight them at 0.3 vs 1.0 for known outcomes. This prevents the noisy labels from dominating.

4. **Production considerations:**
   - More sophisticated methods (EM algorithm, Heckman two-stage) can better handle the selection mechanism
   - The assumed reject default rate (2x accepted) is a heuristic — ideally calibrated from small-scale approval experiments
   - Reject inference should be validated against external data or A/B tests when possible
   - Regulators (SR 11-7, OCC) expect documentation of selection bias treatment in model development